# Disney+ Catalog Analytics and 3NF Relational Database Migration Pipeline
### Exploratory Data Analysis, Data Normalization, & ETL Engineering
---

## Project Overview
Flat files (`.csv`) are highly effective for initial data distribution but are structurally flawed for enterprise software production. They duplicate shared text blocks, allow data anomalies, lack indexing, and explicitly violate the laws of relational database integrity.

This notebook presents a comprehensive **Extract, Transform, Load (ETL)** pipeline that ingests the raw **Disney+ Movies and TV Shows dataset**, executes deep Exploratory Data Analysis (EDA) to map catalog profiles, and programmatically breaks down multi-valued, non-atomic attributes (such as genres, actors, and directors). The final engineering layer automatically normalizes the flat schema into strict **Third Normal Form (3NF)** relational data tables, generating clean, index-ready structures for database streaming.

## Architectural Breakdown & Technical Matrix
The notebook is segmented into distinct data engineering phases designed to show a robust relational toolkit:
1. **Exploratory Data Analysis (EDA):** Profiling content distributions (Movies vs. TV Shows), catalog volume trends across release timelines, and visualizing target audience age ratings using `matplotlib` and `seaborn`.
2. **First Normal Form (1NF) Anomaly Resolution:** Using Python (`pandas`) to parse, split, and explode nested, comma-separated string fields into isolated atomic values.
3. **Third Normal Form (3NF) Relational Migration:** Automatically isolating dimensions and creating mapping keys to establish 4 clean dimension tables and 3 junction (bridge) tables.
4. **Target MySQL Schema Definition:** Designing the complete target DDL SQL structure, enforcing data types, primary keys, explicit indexes, and cascade-on-delete constraints to guarantee strict database relational integrity.

---

## Phase 1: Environment Initialization & Flat File Ingestion
---
**Technical Objective:** Establish the core runtime environment and ingest the raw streaming catalog.

This cell imports the primary analytical libraries (`pandas`) required for handling structured vector operations. It executes a memory-resilient read operation on the `disney_plus_titles.csv` target to index the flat metadata layer into an operational core DataFrame.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/drive/MyDrive/Coding Projects/Datasets/disney_plus_titles.csv')

## Phase 2: Exploratory Data Profiling & Null Attribute Auditing
---
**Objective:** Evaluating missing value frequencies across independent metadata arrays to identify structural data gaps.

Before engineering downstream transformations, this script screens for null constraints. Gaps in attributes like `director` or `country` are identified programmatically to verify which dimensions require specialized formatting or default assignment handling.

In [2]:
# Basic Stats & Missing Values
missing_summary = df.isnull().sum()
type_counts = df['type'].value_counts()

print("--- Missing Values ---")
print(missing_summary)
print("\n--- Type Distribution ---")
print(type_counts)

--- Missing Values ---
show_id           0
type              0
title             0
director        473
cast            190
country         219
date_added        3
release_year      0
rating            3
duration          0
listed_in         0
description       0
dtype: int64

--- Type Distribution ---
type
Movie      1052
TV Show     398
Name: count, dtype: int64


# Phase 3: Visual Analytics — Library Volume & Demographic Distributions
---
**Objective:** Render static, optimized data distributions to profile library assets.

To isolate production patterns, this module uses `matplotlib` and `seaborn` to output explicit volume metrics. It resolves structural deprecation risks by explicitly mapping variables to the `hue` parameters, generating independent distribution plots for both media content styles (`type`) and target age classifications (`rating`).

In [3]:
# Distribution plot
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='type', hue='type', palette='Set2', legend=False) # Fixed here
plt.title('Distribution of Content Types on Disney+')
plt.xlabel('Type')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('disney_type_dist.png')
plt.close()

# Fix for the age ratings plot
plt.figure(figsize=(10, 5))
sns.countplot(
    data=df,
    x='rating',
    hue='rating',
    order=df['rating'].value_counts().index,
    palette='viridis',
    legend=False
)
plt.title('Content Distribution by Age Ratings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('disney_ratings.png')
plt.close()

## Phase 4: First Normal Form (1NF) & Non-Atomic Record Splitting
---
**Objective:** Programmatically resolve First Normal Form (1NF) anomalies by expanding multi-valued nested strings into individual records.

Flat database formats regularly group repeating attributes (like multiple genres or comma-separated cast members) inside a single string cell. This module utilizes specialized text processing to programmatically explode these arrays, ensuring every tuple intersection reflects a strictly atomic value.

## Phase 5: Third Normal Form (3NF) Database Normalization & Module Export
---
**Objective:** Isolate distinct enterprise entities into 4 dimension datasets and 3 bridge junction dataframes to resolve many-to-many properties safely.

To eliminate redundant data blocks, this engine builds independent dimension tracking architectures for `titles`, `genres`, `directors`, and `cast_members`. It simultaneously establishes relational link intersection layers (`title_genres`, `title_directors`, `title_actors`) before exporting the clean data frames out as decoupled relational files.

In [4]:
# --- Programmatic 3NF Normalization Engine (Executes Phases 4 & 5) ---

# 1. Clean and isolate base titles table
titles_table = df[['show_id', 'type', 'title', 'country', 'date_added', 'release_year', 'rating', 'duration', 'description']].copy()
titles_table['date_added'] = titles_table['date_added'].str.strip()

# Helper macro loop to build dimensions and cross-reference mapping records
def generate_relational_layers(dataframe, target_column, unique_id_key, value_label_key):
    # --- PHASE 4 COMPONENT: Explode multi-value items systematically into 1NF ---
    exploded_series = dataframe[['show_id', target_column]].dropna().copy()
    exploded_series[target_column] = exploded_series[target_column].str.split(', ')
    exploded_series = exploded_series.explode(target_column)
    exploded_series[target_column] = exploded_series[target_column].str.strip()

    # --- PHASE 5 COMPONENT: Extract unique labels for 3NF Dimension setup ---
    sorted_uniques = sorted(exploded_series[target_column].unique())
    dimension_dataframe = pd.DataFrame({
        unique_id_key: range(1, len(sorted_uniques) + 1),
        value_label_key: sorted_uniques
    })

    # Establish link intersections for Junction data arrays
    junction_dataframe = exploded_series.merge(dimension_dataframe, left_on=target_column, right_on=value_label_key)
    junction_dataframe = junction_dataframe[['show_id', unique_id_key]].rename(columns={'show_id': 'title_id'})

    return dimension_dataframe, junction_dataframe

# 2. Extract structured dimensions and junction bridges
dim_genres, bridge_genres = generate_relational_layers(df, 'listed_in', 'genre_id', 'genre_name')
dim_directors, bridge_directors = generate_relational_layers(df, 'director', 'director_id', 'director_name')
dim_cast, bridge_cast = generate_relational_layers(df, 'cast', 'actor_id', 'actor_name')

# 3. Export to clean, standalone CSV distribution sheets
titles_table.to_csv('normalized_titles.csv', index=False)
dim_genres.to_csv('dim_genres.csv', index=False)
bridge_genres.to_csv('junction_title_genres.csv', index=False)
dim_directors.to_csv('dim_directors.csv', index=False)
bridge_directors.to_csv('junction_title_directors.csv', index=False)
dim_cast.to_csv('dim_cast.csv', index=False)
bridge_cast.to_csv('junction_title_actors.csv', index=False)

print("✓ Relational normalized tables successfully exported as standalone files!")

✓ Relational normalized tables successfully exported as standalone files!


# Phase 6: Automated DDL SQL Schema Generation
---
**Objective:** Programmatically compile and write out the target MySQL schema blueprints as a standalone database migration asset (`.sql`).

While Python excels at handling the logic of data pipeline transformations, deploying the resulting structures requires explicitly mapping schema rules natively within the host database. This module builds the deployment script to define data types, initialize auto-incrementing primary keys, and enforce relational constraints (`ON DELETE CASCADE`) to guarantee total database integrity.

In [5]:
# --- Programmatic SQL Compiler Layer ---

mysql_schema_script = """-- ======================================================
-- TARGET MYSQL DDL DEPLOYMENT BLUEPRINTS
-- Generated programmatically via Python pipeline
-- ======================================================

CREATE DATABASE IF NOT EXISTS disney_analytics;
USE disney_analytics;

-- 1. Base Core Dimension Table
CREATE TABLE titles (
    show_id VARCHAR(10) PRIMARY KEY,
    type VARCHAR(15) NOT NULL,
    title VARCHAR(255) NOT NULL,
    country VARCHAR(255),
    date_added VARCHAR(50),
    release_year INT NOT NULL,
    rating VARCHAR(10),
    duration VARCHAR(50),
    description TEXT
);

-- 2. Cleaned Atomic Dimension Tables
CREATE TABLE genres (
    genre_id INT AUTO_INCREMENT PRIMARY KEY,
    genre_name VARCHAR(100) NOT NULL UNIQUE
);

CREATE TABLE directors (
    director_id INT AUTO_INCREMENT PRIMARY KEY,
    director_name VARCHAR(150) NOT NULL UNIQUE
);

CREATE TABLE cast_members (
    actor_id INT AUTO_INCREMENT PRIMARY KEY,
    actor_name VARCHAR(150) NOT NULL UNIQUE
);

-- 3. Many-to-Many Relational Junction Bridge Tables
CREATE TABLE title_genres (
    title_id VARCHAR(10),
    genre_id INT,
    PRIMARY KEY (title_id, genre_id),
    FOREIGN KEY (title_id) REFERENCES titles(show_id) ON DELETE CASCADE,
    FOREIGN KEY (genre_id) REFERENCES genres(genre_id) ON DELETE CASCADE
);

CREATE TABLE title_directors (
    title_id VARCHAR(10),
    director_id INT,
    PRIMARY KEY (title_id, director_id),
    FOREIGN KEY (title_id) REFERENCES titles(show_id) ON DELETE CASCADE,
    FOREIGN KEY (director_id) REFERENCES directors(director_id) ON DELETE CASCADE
);

CREATE TABLE title_actors (
    title_id VARCHAR(10),
    actor_id INT,
    PRIMARY KEY (title_id, actor_id),
    FOREIGN KEY (title_id) REFERENCES titles(show_id) ON DELETE CASCADE,
    FOREIGN KEY (actor_id) REFERENCES cast_members(actor_id) ON DELETE CASCADE
);"""

# Programmatically write the script array onto disk
with open("disney_schema_migration.sql", "w") as file:
    file.write(mysql_schema_script)

print(" MySQL DDL deployment script successfully compiled and saved as 'disney_schema_migration.sql'!")

 MySQL DDL deployment script successfully compiled and saved as 'disney_schema_migration.sql'!
